# Dataset 04 | Last.fm API

## 1. Objetivo

Evaluar la API de Last.fm como posible fuente complementaria para el proyecto **Music Recommender | SQL Edition**.

El objetivo principal es investigar si proporciona información útil sobre géneros, etiquetas musicales (*tags*) y similitud entre artistas y canciones.

## 2. Descripción de la fuente

Last.fm es una plataforma musical que recopila información sobre artistas, canciones y hábitos de escucha.

Su API permite consultar determinados datos musicales y relaciones de similitud.

A diferencia de Spotify Tracks 114K y FMA, Last.fm se evaluará como fuente de enriquecimiento del catálogo, no como dataset principal.

## 3. Criterios de evaluación

Se analizarán los siguientes aspectos:

1. **Acceso a la API:** comprobar los requisitos para realizar consultas desde Python.
2. **Información disponible:** identificar datos sobre artistas y canciones.
3. **Etiquetas musicales:** investigar los tags asociados a artistas y canciones.
4. **Similitud musical:** comprobar si existen recomendaciones de artistas o canciones similares.
5. **Calidad de los datos:** revisar la estructura de las respuestas y posibles limitaciones.
6. **Integración con SQL:** valorar cómo almacenar etiquetas y relaciones de similitud.
7. **Utilidad para el recomendador:** determinar qué aporta respecto a Spotify, FMA y MusicBrainz.

## 4. Metodología

Se realizarán consultas de prueba mediante Python y la librería `requests`.

Se utilizará una muestra reducida de artistas y canciones para evaluar las respuestas de la API.

Se analizarán los resultados con Pandas cuando resulte útil.

No se pretende descargar un catálogo completo.

## 5. Resultado esperado

Al finalizar se documentarán:

- Resultados de las consultas.
- Etiquetas musicales disponibles.
- Relaciones de similitud encontradas.
- Posibilidades de integración en SQL.
- Ventajas y limitaciones.
- Valoración final de Last.fm como fuente complementaria.

**Estado: pendiente de exploración.**

In [4]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("LASTFM_API_KEY")

print("API Key cargada:", bool(api_key))

API Key cargada: True


In [5]:
url = "https://ws.audioscrobbler.com/2.0/"

params = {
    "method": "artist.getTopTags",
    "artist": "BTS",
    "api_key": api_key,
    "format": "json"
}

response = requests.get(
    url,
    params=params,
    timeout=15
)

print("Código de respuesta:", response.status_code)

if response.status_code == 200:
    lastfm_data = response.json()

    if "error" in lastfm_data:
        print("Error de Last.fm:", lastfm_data["message"])
    else:
        tags_df = pd.DataFrame(lastfm_data["toptags"]["tag"])
        display(tags_df.head(10))
else:
    print("Error en la consulta")

Código de respuesta: 200


,count,name,url
0,100,k-pop,https://www.last.fm/tag/k-pop
1,60,bts,https://www.last.fm/tag/bts
2,36,Kpop,https://www.last.fm/tag/Kpop
3,34,Korean,https://www.last.fm/tag/Korean
4,17,pop,https://www.last.fm/tag/pop
5,13,rap,https://www.last.fm/tag/rap
6,9,hip hop,https://www.last.fm/tag/hip+hop
7,2,Hip-Hop,https://www.last.fm/tag/Hip-Hop
8,2,rnb,https://www.last.fm/tag/rnb
9,2,dance,https://www.last.fm/tag/dance


In [6]:
print("Etiquetas recibidas:", len(tags_df))

print("\nValores nulos:")
print(tags_df.isna().sum())

print("\nNombres duplicados exactos:")
print(tags_df["name"].duplicated().sum())

Etiquetas recibidas: 10

Valores nulos:
count    0
name     0
url      0
dtype: int64

Nombres duplicados exactos:
0


In [7]:
params = {
    "method": "artist.getSimilar",
    "artist": "BTS",
    "api_key": api_key,
    "format": "json",
    "limit": 10
}

response = requests.get(
    url,
    params=params,
    timeout=15
)

print("Código de respuesta:", response.status_code)

if response.status_code == 200:
    similar_data = response.json()

    if "error" in similar_data:
        print("Error de Last.fm:", similar_data["message"])
    else:
        similar_df = pd.DataFrame(
            similar_data["similarartists"]["artist"]
        )

        display(similar_df[["name", "match"]])
else:
    print("Error en la consulta")

Código de respuesta: 200


,name,match
0,RM,1
1,Agust D,0.970091
2,Jimin,0.675175
3,j-hope,0.613650
4,"BTS, Halsey",0.588091
5,Jung Kook,0.580266
6,V,0.517417
7,방탄소년단,0.476684
8,Jin,0.422672
9,"Coldplay, BTS",0.366259


In [8]:
print("Artistas similares:", len(similar_df))

print("Nombres únicos:", similar_df["name"].nunique())

print("\nValores nulos:")
print(similar_df[["name", "match"]].isna().sum())

print("\nRango de similitud:")
print(pd.to_numeric(similar_df["match"]).agg(["min", "max"]))

Artistas similares: 10
Nombres únicos: 10

Valores nulos:
name     0
match    0
dtype: int64

Rango de similitud:
min    0.366259
max    1.000000
Name: match, dtype: float64


## Last.fm API | Resultados y conclusiones

### 1. Objetivo de la exploración

Evaluar Last.fm como fuente complementaria de etiquetas musicales y relaciones de similitud para el proyecto **Music Recommender | SQL Edition**.

### 2. Metodología

Se ha utilizado Python con las librerías `requests`, `pandas` y `python-dotenv`.

La API Key se ha almacenado en un archivo `.env`, excluido del control de versiones mediante `.gitignore`.

Se han realizado consultas utilizando BTS como artista de prueba.

### 3. Consulta de etiquetas musicales

Se ha utilizado el método `artist.getTopTags`.

**Resultados:**

- Etiquetas recibidas: 10.
- Valores nulos: 0.
- Nombres duplicados exactos: 0.

Entre las etiquetas encontradas destacan:

| Etiqueta | Peso |
|---|---:|
| k-pop | 100 |
| bts | 60 |
| Kpop | 36 |
| Korean | 34 |
| pop | 17 |
| rap | 13 |
| hip hop | 9 |
| Hip-Hop | 2 |
| rnb | 2 |
| dance | 2 |

Se han detectado variantes de escritura como `k-pop` y `Kpop`, o `hip hop` y `Hip-Hop`.

También se ha observado que no todas las etiquetas representan géneros musicales.

**Conclusión:** será necesario normalizar y clasificar las etiquetas antes de utilizarlas en SQL.

### 4. Consulta de artistas similares

Se ha utilizado el método `artist.getSimilar`.

La API ha devuelto artistas relacionados con BTS y una puntuación de similitud denominada `match`.

**Resultados:**

- Artistas similares recibidos: 10.
- Nombres únicos: 10.
- Valores nulos: 0.
- Similitud mínima: 0,366259.
- Similitud máxima: 1,000000.

Entre los resultados aparecen RM, Agust D, Jimin, j-hope, Jung Kook, V y Jin.

También aparecen nombres alternativos y créditos de colaboraciones.

### 5. Calidad y normalización de datos

La muestra analizada no presenta valores nulos ni nombres duplicados exactos.

Sin embargo, se han identificado posibles problemas:

- Variantes de escritura de un mismo género.
- Diferentes nombres para un mismo artista.
- Créditos de colaboraciones tratados como nombres de artista.
- Etiquetas que no representan géneros musicales.

Estos problemas deberán considerarse durante la limpieza y normalización.

### 6. Posible integración en SQL

Last.fm podría aportar información para las siguientes tablas:

- `tags`: catálogo de etiquetas musicales.
- `artist_tags`: relación entre artistas y etiquetas.
- `artist_similarity`: relaciones de similitud entre artistas.

Las etiquetas permitirían clasificar artistas por estilos y otras características.

Las relaciones de similitud podrían utilizarse para generar recomendaciones.

### 7. Ventajas y limitaciones

**Ventajas:**

- Acceso a etiquetas musicales.
- Relaciones de similitud entre artistas.
- Información útil para recomendaciones basadas en afinidad.
- Integración mediante peticiones HTTP y respuestas JSON.
- Posibilidad de complementar otras fuentes musicales.

**Limitaciones:**

- Requiere una API Key.
- Las etiquetas necesitan normalización.
- Las relaciones de similitud pueden incluir nombres alternativos y colaboraciones.
- Las puntuaciones de similitud proceden del servicio y no deben interpretarse como medidas objetivas de calidad musical.
- La evaluación se ha realizado sobre una muestra reducida.

### 8. Conclusión

Last.fm presenta potencial como fuente complementaria para enriquecer el Music Recommender.

Sus etiquetas y relaciones de similitud podrían ayudar a desarrollar recomendaciones basadas en géneros, afinidades y artistas relacionados.

No se considera necesario utilizar Last.fm como fuente principal del catálogo.

**Valoración provisional:** utilizar Last.fm como posible complemento para etiquetas y similitudes musicales.

**Estado: exploración inicial de Last.fm completada.**